# Семинар 8. Transformer OCR

https://huggingface.co/docs/transformers/model_doc/trocr 

In [12]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import requests
from PIL import Image
import os
import sys
import time
import json
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import easyocr

# Настройка отображения matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# load image from the IAM dataset
url = "https://fki.tic.heia-fr.ch/static/img/a01-122-02.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


industry , " Mr. Brown commented icily . " Let us have a


## Задание 1. Запустите распознавание собственного документа и сравните результат с easyocr из 7 лабораторной. 

In [13]:
class EasyOCRDemo:
    """Класс для демонстрации возможностей OCR"""
    
    def __init__(self, languages: List[str] = ['ru', 'en'], gpu: bool = False):
        """
        Инициализация OCR-читателя
        
        Args:
            languages: Список языков для распознавания
            gpu: Использовать GPU
        """
        print(f"Инициализация OCR с языками: {languages}")
        print(f"Режим: {'GPU' if gpu else 'CPU'}")
        
        try:
            self.reader = easyocr.Reader(languages, gpu=gpu)
            self.languages = languages
            print("OCR успешно инициализирован")
        except Exception as e:
            print(f"Ошибка: {e}")
            raise
    
    def recognize(self, image_path: str, 
                 detail: int = 1,
                 paragraph: bool = False) -> List:
        """
        Распознавание текста на изображении
        """
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Файл не найден: {image_path}")
        
        print(f"\n📄 Анализ: {os.path.basename(image_path)}")
        start = time.time()
        
        result = self.reader.readtext(
            image_path,
            detail=detail,
            paragraph=paragraph,
            width_ths=0.7,
            height_ths=0.5
        )
        
        elapsed = time.time() - start
        print(f"✅ Готово за {elapsed:.2f} сек")
        
        return result
    
    def visualize(self, image_path: str, result: List = None,
                  show_confidence: bool = True, figsize=(15, 10)):
        """
        Визуализация результатов распознавания
        """
        if result is None:
            result = self.recognize(image_path)
        
        # Загружаем изображение
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Создаем фигуру
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        # Оригинальное изображение
        ax1.imshow(img)
        ax1.set_title('Оригинальное изображение')
        ax1.axis('off')
        
        # Изображение с разметкой
        ax2.imshow(img)
        ax2.set_title(f'Распознано объектов: {len(result)}')
        
        # Цвета для разных уровней уверенности
        colors = {'high': 'green', 'medium': 'orange', 'low': 'red'}
        
        for detection in result:
            points = detection[0]
            text = detection[1]
            conf = detection[2]
            
            # Определяем цвет
            if conf > 0.8:
                color = colors['high']
            elif conf > 0.5:
                color = colors['medium']
            else:
                color = colors['low']
            
            # Рисуем bounding box
            polygon = plt.Polygon(points, fill=False, 
                                 edgecolor=color, linewidth=2)
            ax2.add_patch(polygon)
            
            # Добавляем текст
            x, y = points[0]
            if show_confidence:
                label = f"{text} ({conf:.2f})"
            else:
                label = text
            
            ax2.text(x, y-5, label, fontsize=8,
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
        
        ax2.axis('off')
        plt.tight_layout()
        plt.show()
        
        return result
    
    def get_text(self, result: List, min_confidence: float = 0.0) -> str:
        """
        Извлечение текста из результатов
        """
        texts = []
        for detection in result:
            if detection[2] >= min_confidence:
                texts.append(detection[1])
        return '\n'.join(texts)
    
    def save_results(self, result: List, output_path: str):
        """
        Сохранение результатов в JSON
        """
        serializable = []
        for item in result:
            serializable.append({
                'bbox': [[float(x) for x in point] for point in item[0]],
                'text': item[1],
                'confidence': float(item[2])
            })
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(serializable, f, ensure_ascii=False, indent=2)
        
        print(f"💾 Результаты сохранены в {output_path}")

# Создаем экземпляр
ocr = EasyOCRDemo(languages=['ru', 'en'], gpu=True)

Инициализация OCR с языками: ['ru', 'en']
Режим: GPU
OCR успешно инициализирован


In [ ]:
def _levenshtein_distance(a, b):
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            ins = cur[j - 1] + 1
            delete = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            cur.append(min(ins, delete, sub))
        prev = cur
    return prev[-1]

def _normalize_text(s):
    return " ".join(s.lower().split())

def compare_ocr_results(trocr_text, easyocr_text):
    t = _normalize_text(trocr_text)
    e = _normalize_text(easyocr_text)
    if not t and not e:
        return {
            "cer": 0.0,
            "cer_accuracy": 1.0,
            "wer": 0.0,
            "wer_accuracy": 1.0,
            "trocr_len": 0,
            "easyocr_len": 0,
            "note": "both_empty",
        }
    cer_dist = _levenshtein_distance(t, e)
    cer_den = max(1, max(len(t), len(e)))
    cer = cer_dist / cer_den
    cer_acc = 1.0 - cer

    t_words = t.split()
    e_words = e.split()
    wer_dist = _levenshtein_distance(t_words, e_words)
    wer_den = max(1, max(len(t_words), len(e_words)))
    wer = wer_dist / wer_den
    wer_acc = 1.0 - wer

    return {
        "cer": cer,
        "cer_accuracy": cer_acc,
        "wer": wer,
        "wer_accuracy": wer_acc,
        "trocr_len": len(t),
        "easyocr_len": len(e),
        "note": "normalized_by_max_len",
    }

# Example usage on the same image:

In [16]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import requests
from PIL import Image

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# load image from the IAM dataset
real_image = "data/twd10.png"
image = Image.open(real_image).convert("RGB")

pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


1 . Women's Day is celebrated on March 8 .


In [18]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

# load image from the IAM dataset
real_image = "data/twd11.png"
image = Image.open(real_image).convert("RGB")

pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


THAT BOTH ARE BRANCHES OF A NONGOLOID STOCK WHICH


In [ ]:
# EasyOCR detects lines -> TrOCR reads each line
import numpy as np
from PIL import Image

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

real_image = "data/twd11.jpg"
img_pil = Image.open(real_image).convert("RGB")
img = np.array(img_pil)

import easyocr
reader = easyocr.Reader(['ru','en'], gpu=False)
detections = reader.readtext(img, detail=1)

boxes = []
for r in detections:
    bbox = r[0]
    xs = [int(p[0]) for p in bbox]
    ys = [int(p[1]) for p in bbox]
    x1, x2 = max(0, min(xs)), min(img.shape[1], max(xs))
    y1, y2 = max(0, min(ys)), min(img.shape[0], max(ys))
    if (y2 - y1) > 6 and (x2 - x1) > 6:
        boxes.append((x1, y1, x2, y2))

# Sort boxes by lines using adaptive y-tolerance
def sort_boxes_by_lines(boxes):
    if not boxes:
        return []
    heights = [b[3] - b[1] for b in boxes]
    med_h = float(np.median(heights)) if heights else 0
    y_tol = max(8, int(med_h * 0.6))
    items = sorted(boxes, key=lambda b: (b[1], b[0]))
    lines = []
    for (x1, y1, x2, y2) in items:
        yc = (y1 + y2) / 2
        placed = False
        for line in lines:
            if abs(yc - line['yc']) <= y_tol:
                line['boxes'].append((x1, y1, x2, y2))
                line['yc'] = (line['yc'] * line['n'] + yc) / (line['n'] + 1)
                line['n'] += 1
                placed = True
                break
        if not placed:
            lines.append({'yc': yc, 'n': 1, 'boxes': [(x1, y1, x2, y2)]})
    lines.sort(key=lambda l: l['yc'])
    ordered = []
    for line in lines:
        line['boxes'].sort(key=lambda b: b[0])
        ordered.extend(line['boxes'])
    return ordered

boxes = sort_boxes_by_lines(boxes)

lines = []
for (x1, y1, x2, y2) in boxes:
    pad = 4
    x1c = max(0, x1 - pad)
    y1c = max(0, y1 - pad)
    x2c = min(img.shape[1], x2 + pad)
    y2c = min(img.shape[0], y2 + pad)
    crop = img[y1c:y2c, x1c:x2c]
    crop_pil = Image.fromarray(crop)
    pixel_values = processor(crop_pil, return_tensors="pt").pixel_values
    gen_ids = model.generate(pixel_values)
    txt = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    if txt:
        lines.append(txt)

generated_text = '\n'.join(lines)
print(generated_text)
print('Detected lines:', len(lines))

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using CPU. Note: This module is much faster with a GPU.


One page handwriting
Human beings .
are
superior
to all other .
living
gheing's
because they
can .
speak .
speaking
telling
to
someone .
miho
is
listening .
to
you
in
the
same
sense
is
called
communication .
we
ican .
convey .
our
message
to
person .
Detected lines: 33
